# Stage 00: ground-truth index

Builds `V-B intersect V-P`, the index every later stage verifies against.

- **V-B** is the Enron maildir under the message-level filters, with quote and
  forward stripping **disabled**. The Pile's Enron component keeps quoted text,
  so stripping would discard addresses a model could genuinely have memorized.
- **V-P** is the Pile's Enron Emails component, which is the corpus the models
  actually trained on.

Intersecting gives an index where every value is provably in the training data
and bot traffic has still been filtered out.

Nothing here needs a GPU. Expect roughly 40 minutes: about 15 for the Pile
download and 10 for the maildir pass.

In [ ]:
import os
import sys
from pathlib import Path

# PYTHONHASHSEED must be exported before the interpreter starts to have any
# effect. Setting it here would do nothing, so instead we assert that nothing
# in this project depends on it: every seed is derived with zlib.crc32.
ROOT = Path.cwd()
while not (ROOT / "src" / "dea_a3").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from dea_a3.config import CONFIG
from dea_a3.paths import Paths

PATHS = Paths(ROOT)
PATHS.ensure()
print("root      :", ROOT)
print("study     :", CONFIG.study)
print("index     :", CONFIG.index_variant)

In [ ]:
# Preflight. Every notebook checks its inputs before doing any work; round 10
# of the earlier study produced two wrong conclusions by running against a
# half-populated tree.
def preflight(required_paths, required_packages=("numpy", "pandas", "pyarrow", "scipy")):
    ok = True
    for label, path in required_paths:
        exists = Path(path).exists()
        print(f"[{'PASS' if exists else 'FAIL'}] {label}: {path}")
        ok &= exists
    for pkg in required_packages:
        try:
            module = __import__(pkg)
            print(f"[PASS] {pkg} {getattr(module, '__version__', '?')}")
        except ImportError:
            print(f"[FAIL] {pkg} not importable")
            ok = False
    if not ok:
        raise SystemExit("preflight failed; fix the above before continuing")
    print("preflight OK")

In [ ]:
preflight([
    ("maildir", PATHS.maildir),
    ("data dir", PATHS.data),
])

## 1. The Pile's Enron component

Streams every shard, keeps only records tagged `Enron Emails`, and caches them. Resumable: a shard that already finished is skipped.

In [ ]:
from dea_a3 import pile

paths_out = pile.extract_all(
    repo=CONFIG.pile_repo,
    n_shards=CONFIG.pile_shards,
    set_name=CONFIG.pile_set_name,
    out_dir=PATHS.pile_cache,
    workers=CONFIG.pile_workers,
)
print(f"shards cached: {len(paths_out)}")

In [ ]:
pile_counts, n_pile_docs = pile.build_value_counts(PATHS.pile_cache, kinds=CONFIG.kinds)
for kind in CONFIG.kinds:
    print(f"{kind:6s} distinct in Pile: {len(pile_counts[kind]):,}")
print(f"Pile Enron documents: {n_pile_docs:,}")

## 2. The maildir side (V-B)

Quote stripping is off. The automated-sender and broadcast filters stay on.

In [ ]:
from dea_a3 import corpus

records, corpus_stats = corpus.load_records(
    PATHS.maildir,
    strip=CONFIG.strip_quoted,
    broadcast_limit=CONFIG.broadcast_recipient_limit,
)
for key in sorted(corpus_stats):
    print(f"{key:32s} {corpus_stats[key]:>8,}")

In [ ]:
maildir_values = corpus.collect_values(records, kinds=CONFIG.kinds)
for kind in CONFIG.kinds:
    print(f"{kind:6s} distinct in maildir (V-B): {len(maildir_values[kind]):,}")

## 3. Coverage and precision

This reproduces the variant comparison inside the study, so the index-choice claim rests on an artifact this pipeline produced rather than on a number quoted from a report.

In [ ]:
from dea_a3 import index

variant_table = index.variant_stats(maildir_values, pile_counts, kinds=CONFIG.kinds)
variant_table.to_csv(PATHS.index_dir / "variant_stats.csv", index=False)
variant_table

## 4. Build and persist the index

Only salted hashes are written. No plaintext value reaches disk at any point.

In [ ]:
from dea_a3 import pii

salt = pii.load_or_create_salt(PATHS.salt_file)
index_frame = index.build_index(maildir_values, pile_counts, salt, kinds=CONFIG.kinds)
index_frame.to_parquet(PATHS.index_file, index=False)

summary = index.index_summary(index_frame)
summary.to_csv(PATHS.index_dir / "index_summary.csv", index=False)
print(f"index rows: {len(index_frame):,}")
summary

In [ ]:
# Guard: assert no plaintext leaked into the artifact.
blob = index_frame.to_csv(index=False)
assert "@" not in blob, "plaintext email reached the index file"
print("no-plaintext check passed")

## 5. True-prefix pool

The first twelve words of documents that contain at least one indexed value. Only prefixes are stored, and a prefix that itself contains PII is discarded so the attacker's own input can never be scored as a leak.

In [ ]:
import pandas as pd
from dea_a3 import prompts as prompt_mod

known = index.hashes_by_kind(index_frame)
pool = []
for rec in records:
    text = rec.indexed_text()
    found = pii.extract_values(text, kinds=CONFIG.kinds)
    if not any(
        pii.salted_hash(kind, value, salt) in known.get(kind, set())
        for kind in CONFIG.kinds for value in found[kind]
    ):
        continue
    prefix = prompt_mod.make_true_prefix(rec.body, CONFIG.true_prefix_words)
    if prefix is None:
        continue
    if any(pii.extract_values(prefix, kinds=CONFIG.kinds)[k] for k in CONFIG.kinds):
        continue
    pool.append(prefix)

pool = sorted(set(pool))
pd.DataFrame({"prefix": pool}).to_parquet(PATHS.true_prefix_file, index=False)
print(f"true-prefix pool: {len(pool):,}")

In [ ]:
assert len(pool) >= 1000, "true-prefix pool is too small; check the maildir path"
for prefix in pool[:3]:
    print(repr(prefix[:70]))

## 6. Manifest

In [ ]:
from dea_a3 import manifest

manifest.write_manifest(
    PATHS.index_dir, "00_index", CONFIG.to_dict(), ROOT,
    extra={
        "corpus_stats": dict(corpus_stats),
        "pile_documents": n_pile_docs,
        "index_rows": int(len(index_frame)),
        "true_prefix_pool": int(len(pool)),
    },
)
print("stage 00 complete")